# Hyperspectral Statistical Ensemble Benchmark

Ensemble of Global RX + Local RX with CDF normalization and score
fusion on all 4 PRISMA L2D benchmark scenes.
Both detectors run independently, scores are CDF-normalized to [0,1],
then fused via product strategy for the final anomaly map.

### 1. Load all 4 benchmark scenes

In [ ]:
import os
import sys
import pathlib

import numpy as np
import rasterio
import matplotlib.pyplot as plt

PROJECT_ROOT = str(pathlib.Path(os.getcwd()).parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app.models.file_processing.sources import FileSourceConfig
from app.utils.dataset_builder.prisma_dataset_builder import PrismaDatasetBuilder
from app.models.dataset.vendables import BandFilterConfig, DEFAULT_COMMON_WAVELENGTH_GRID

BENCHMARK_ROOT = os.path.join(PROJECT_ROOT, "benchmarking", "hyperspectral")
DATASET_IDS = [1, 2, 3, 4]

band_filter_config = BandFilterConfig(common_wavelength_grid=DEFAULT_COMMON_WAVELENGTH_GRID)

datasets = {}
for did in DATASET_IDS:
    folder = os.path.join(BENCHMARK_ROOT, str(did))
    he5_files = [f for f in os.listdir(folder) if f.endswith(".he5")]
    assert len(he5_files) == 1, f"Expected 1 HE5 in {folder}, found {he5_files}"
    he5_path = os.path.join(folder, he5_files[0])
    gt_path = os.path.join(folder, "gt.tif")

    print(f"=== Loading Dataset {did}: {he5_files[0][:50]}... ===")

    builder = PrismaDatasetBuilder(
        file_source_configuration=FileSourceConfig(source_path=he5_path)
    )
    vendable = builder.vend_dataset(band_filter_config=band_filter_config)

    with rasterio.open(gt_path) as src:
        gt_data = src.read(1)

    cube = vendable.normalized_hyperspectral_cube
    validity = vendable.validity_cube
    wavelengths = vendable.band_cw_order
    band_validity = vendable.band_validity_by_position

    # After pipeline filtering, pixels are either fully valid or fully invalid.
    spatial_validity = (validity.sum(axis=0) > 0).astype(np.uint8)

    datasets[did] = {
        "name": he5_files[0].replace(".he5", ""),
        "vendable": vendable,
        "cube": cube,
        "validity": validity,
        "spatial_validity": spatial_validity,
        "wavelengths": np.array(wavelengths),
        "band_validity": np.array(band_validity),
        "gt": (gt_data > 0).astype(np.uint8),
    }

    C, H, W = cube.shape
    n_anom = datasets[did]["gt"].sum()
    valid_frac = spatial_validity.sum() / (H * W) * 100
    print(f"  Cube: {cube.shape}, Valid bands: {C}")
    print(f"  Spatial validity: {valid_frac:.1f}%, Anomaly pixels: {n_anom:,}")
    print()

### 2. RGB Composites (2x2)

In [ ]:
def make_rgb_composite(cube, wavelengths, validity, r_nm=650, g_nm=550, b_nm=450):
    """Create an RGB composite from the nearest bands to target wavelengths."""
    r_idx = np.argmin(np.abs(wavelengths - r_nm))
    g_idx = np.argmin(np.abs(wavelengths - g_nm))
    b_idx = np.argmin(np.abs(wavelengths - b_nm))
    rgb = np.stack([cube[r_idx], cube[g_idx], cube[b_idx]], axis=-1)
    for ch in range(3):
        valid_vals = rgb[:, :, ch][validity > 0]
        if len(valid_vals) > 0:
            p2, p98 = np.percentile(valid_vals, [2, 98])
            rgb[:, :, ch] = np.clip((rgb[:, :, ch] - p2) / (p98 - p2 + 1e-10), 0, 1)
    rgb[validity == 0] = 0
    return rgb

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    rgb = make_rgb_composite(d["cube"], d["wavelengths"], d["spatial_validity"])
    axes[idx].imshow(rgb)
    axes[idx].set_title(f"Scene {did}: {d['name'][:40]}...", fontsize=10)
    axes[idx].axis("off")
plt.suptitle("RGB Composites (R=650nm, G=550nm, B=450nm)", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### 3. Anomaly Overlay on RGB (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    rgb = make_rgb_composite(d["cube"], d["wavelengths"], d["spatial_validity"])
    axes[idx].imshow(rgb)
    anom_rows, anom_cols = np.where(d["gt"] > 0)
    axes[idx].scatter(anom_cols, anom_rows, c="cyan", s=8, marker="o",
                      linewidths=0.8, edgecolors="cyan", alpha=0.9,
                      label=f"Anomalies ({len(anom_rows):,} px)")
    axes[idx].set_title(f"Scene {did} — {len(anom_rows):,} anomaly pixels", fontsize=11)
    axes[idx].axis("off")
    axes[idx].legend(loc="lower right", fontsize=9, markerscale=2,
                     framealpha=0.8, facecolor="black", labelcolor="cyan")
plt.suptitle("Anomaly Ground Truth on RGB Composites", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### 4. Zoomed Anomaly Regions (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()
PAD = 200
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    gt = d["gt"]; H, W = gt.shape
    anom_rows, anom_cols = np.where(gt > 0)
    if len(anom_rows) == 0:
        axes[idx].set_title(f"Scene {did} — no anomalies"); continue
    r_min, r_max = max(0, anom_rows.min()-PAD), min(H, anom_rows.max()+PAD)
    c_min, c_max = max(0, anom_cols.min()-PAD), min(W, anom_cols.max()+PAD)
    crop_cube = d["cube"][:, r_min:r_max, c_min:c_max]
    crop_valid = d["spatial_validity"][r_min:r_max, c_min:c_max]
    crop_gt = gt[r_min:r_max, c_min:c_max]
    rgb = make_rgb_composite(crop_cube, d["wavelengths"], crop_valid)
    axes[idx].imshow(rgb)
    ar, ac = np.where(crop_gt > 0)
    axes[idx].scatter(ac, ar, c="cyan", s=12, marker="o", linewidths=1.0,
                      edgecolors="cyan", alpha=0.9)
    axes[idx].set_title(f"Scene {did} — zoomed ({r_max-r_min}x{c_max-c_min} px)", fontsize=11)
    axes[idx].axis("off")
plt.suptitle("Zoomed Anomaly Regions (200px padding)", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### 5. Summary Statistics

In [ ]:
print(f"{'':=<110}")
print(f"{'HYPERSPECTRAL BENCHMARK DATASET SUMMARY':^110}")
print(f"{'':=<110}")
for did in DATASET_IDS:
    d = datasets[did]
    cube, gt, sv, wl = d["cube"], d["gt"], d["spatial_validity"], d["wavelengths"]
    bv = d["band_validity"]
    C, H, W = cube.shape
    n_anom = gt.sum()
    n_anom_valid = gt[sv == 1].sum()
    valid_frac = sv.sum() / (H * W) * 100
    print(f"\n--- Dataset {did}: {d['name'][:60]} ---")
    print(f"  Dimensions: {H}x{W}, Bands: {C} ({bv.sum()} valid)")
    print(f"  Wavelengths: [{wl.min():.1f}, {wl.max():.1f}] nm")
    print(f"  Valid pixels: {sv.sum():,}/{H*W:,} ({valid_frac:.1f}%)")
    print(f"  Anomalies: {n_anom:,} ({n_anom_valid:,} on valid pixels)")
    if n_anom > 0:
        print(f"  Prevalence: {n_anom_valid / sv.sum() * 100:.4f}%")

---
## Ensemble (GRX + LRX) Anomaly Detection

### Run Ensemble on All 4 Datasets

In [ ]:
from app.detectors.statistical_ensembler import StatisticalEnsembler

for did in DATASET_IDS:
    d = datasets[did]
    print(f"=== Running Ensemble (GRX+LRX) on Dataset {did} ===")

    detector = StatisticalEnsembler(vendable=d["vendable"])
    detector.fit()

    score_map = detector.detect(d["cube"], d["validity"])

    mask = np.isfinite(score_map).astype(float)
    scores_clean = np.nan_to_num(score_map, nan=0.0)

    d["mask"] = mask
    d["residual"] = scores_clean * mask

    valid_scores = scores_clean[mask == 1]
    print(f"  Valid pixels: {int(mask.sum()):,}")
    print(f"  Score range: [{valid_scores.min():.4f}, {valid_scores.max():.4f}]")
    print(f"  Mean={valid_scores.mean():.4f}, Median={np.median(valid_scores):.4f}")
    print()

### ROC & F1 Sweep

In [ ]:
from sklearn.metrics import roc_curve, auc

PERCENTILE_RANGE = np.arange(80, 100.0, 1)

for did in DATASET_IDS:
    d = datasets[did]
    residual = d["residual"]
    mask = d["mask"]
    gt = d["gt"]

    valid_mask_flat = mask.astype(bool).ravel()
    scores = residual.ravel()[valid_mask_flat]
    gt_labels = gt.ravel()[valid_mask_flat].astype(int)

    fpr, tpr, _ = roc_curve(gt_labels, scores)
    roc_auc = auc(fpr, tpr)

    precisions, recalls, f1_scores, thresholds_at_pct = [], [], [], []
    for p in PERCENTILE_RANGE:
        thresh = np.percentile(scores, p)
        predicted = (scores > thresh).astype(int)
        tp = ((predicted == 1) & (gt_labels == 1)).sum()
        fp = ((predicted == 1) & (gt_labels == 0)).sum()
        fn = ((predicted == 0) & (gt_labels == 1)).sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
        precisions.append(precision); recalls.append(recall)
        f1_scores.append(f1); thresholds_at_pct.append(thresh)

    precisions = np.array(precisions); recalls = np.array(recalls)
    f1_scores = np.array(f1_scores); thresholds_at_pct = np.array(thresholds_at_pct)
    best_idx = np.argmax(f1_scores)

    d["fpr"] = fpr; d["tpr"] = tpr; d["roc_auc"] = roc_auc
    d["precisions"] = precisions; d["recalls"] = recalls; d["f1_scores"] = f1_scores
    d["best_idx"] = best_idx; d["best_pct"] = PERCENTILE_RANGE[best_idx]
    d["best_thresh"] = thresholds_at_pct[best_idx]
    d["best_f1"] = f1_scores[best_idx]
    d["best_precision"] = precisions[best_idx]; d["best_recall"] = recalls[best_idx]
    d["scores"] = scores; d["gt_labels"] = gt_labels

    p9995_thresh = np.percentile(scores, 99.95)
    d["detected_binary"] = ((residual > p9995_thresh) & (mask == 1)).astype(np.uint8)

    print(f"Dataset {did}: AUC={roc_auc:.4f}, Best F1={f1_scores[best_idx]:.4f} "
          f"@ P{PERCENTILE_RANGE[best_idx]:.0f}, "
          f"Prec={precisions[best_idx]:.4f}, Rec={recalls[best_idx]:.4f}")

### Anomaly Score Maps (clipped at P99.5)

In [ ]:
CLIP_PERCENTILE = 99.5

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    residual = d["residual"]; mask = d["mask"]
    valid_residuals = residual[mask == 1]
    clip_val = np.percentile(valid_residuals, CLIP_PERCENTILE)
    residual_masked = np.ma.masked_where(mask == 0, residual)
    im = axes[idx].imshow(residual_masked, cmap="Reds", vmin=0, vmax=clip_val)
    axes[idx].set_title(f"Scene {did} — Score (clipped P{CLIP_PERCENTILE}={clip_val:.2f})", fontsize=10)
    axes[idx].axis("off")
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
plt.suptitle("Anomaly Scores — Clipped at 99.5th Percentile", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### Detected vs Ground Truth

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    rgb = make_rgb_composite(d["cube"], d["wavelengths"], d["spatial_validity"])
    axes[idx].imshow(rgb)
    gt_rows, gt_cols = np.where(d["gt"] > 0)
    axes[idx].scatter(gt_cols, gt_rows, c="cyan", s=10, marker="o",
                      linewidths=1.0, edgecolors="cyan", alpha=0.9,
                      label=f"GT ({len(gt_rows):,} px)")
    det_rows, det_cols = np.where(d["detected_binary"] > 0)
    axes[idx].scatter(det_cols, det_rows, c="lime", s=6, marker="s",
                      linewidths=0.5, edgecolors="lime", alpha=0.8,
                      label=f"Detected ({len(det_rows):,} px)")
    axes[idx].set_title(f"Scene {did} — F1={d['best_f1']:.4f}", fontsize=11)
    axes[idx].axis("off")
    axes[idx].legend(loc="lower right", fontsize=9, markerscale=2,
                     framealpha=0.8, facecolor="black", labelcolor="white")
plt.suptitle("Detected (green) vs Ground Truth (cyan) Anomalies", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    axes[idx].plot(d["fpr"], d["tpr"], color="darkorange", lw=2,
                   label=f"ROC (AUC = {d['roc_auc']:.4f})")
    axes[idx].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random")
    axes[idx].set_xlim([0, 1]); axes[idx].set_ylim([0, 1.05])
    axes[idx].set_xlabel("False Positive Rate"); axes[idx].set_ylabel("True Positive Rate")
    axes[idx].set_title(f"Scene {did} — AUC = {d['roc_auc']:.4f}", fontsize=11)
    axes[idx].legend(loc="lower right", fontsize=9)
    axes[idx].grid(True, alpha=0.3)
plt.suptitle("ROC Curves", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### Precision, Recall & F1 vs Percentile

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()
for idx, did in enumerate(DATASET_IDS):
    d = datasets[did]
    axes[idx].plot(PERCENTILE_RANGE, d["precisions"], color="blue", lw=2, label="Precision")
    axes[idx].plot(PERCENTILE_RANGE, d["recalls"], color="red", lw=2, label="Recall")
    axes[idx].plot(PERCENTILE_RANGE, d["f1_scores"], color="green", lw=2, label="F1")
    axes[idx].axvline(d["best_pct"], color="gray", linestyle="--", alpha=0.7,
                      label=f"Best F1={d['best_f1']:.4f} @ P{d['best_pct']:.0f}")
    axes[idx].set_xlabel("Percentile Threshold"); axes[idx].set_ylabel("Score")
    axes[idx].set_title(f"Scene {did}", fontsize=11)
    axes[idx].legend(fontsize=8); axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xlim([PERCENTILE_RANGE[0], PERCENTILE_RANGE[-1]])
    axes[idx].set_ylim([0, 1.05])
plt.suptitle("Precision, Recall & F1 vs Percentile Threshold", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### Summary Table

In [ ]:
print(f"{'':=<110}")
print(f"{'BENCHMARK RESULTS: Statistical Ensemble (GRX+LRX)':^110}")
print(f"{'':=<110}")
print()
print(f"{'Metric':<30} {'Scene 1':>15} {'Scene 2':>15} {'Scene 3':>15} {'Scene 4':>15}")
print(f"{'-'*90}")

for label, key_fn in [
    ("AUC-ROC", lambda d: f"{d['roc_auc']:.4f}"),
    ("Best F1", lambda d: f"{d['best_f1']:.4f}"),
    ("Best Precision", lambda d: f"{d['best_precision']:.4f}"),
    ("Best Recall", lambda d: f"{d['best_recall']:.4f}"),
    ("Best Percentile", lambda d: f"P{d['best_pct']:.0f}"),
    ("Best Threshold", lambda d: f"{d['best_thresh']:.4f}"),
    ("GT Anomaly Pixels", lambda d: f"{d['gt'].sum():,}"),
    ("Detected Pixels", lambda d: f"{d['detected_binary'].sum():,}"),
    ("Valid Pixels", lambda d: f"{(d['mask'] == 1).sum():,}"),
    ("Mean Score", lambda d: f"{d['residual'][d['mask'] == 1].mean():.4f}"),
    ("Median Score", lambda d: f"{np.median(d['residual'][d['mask'] == 1]):.4f}"),
    ("P99.5 Score", lambda d: f"{np.percentile(d['residual'][d['mask'] == 1], 99.5):.4f}"),
    ("Max Score", lambda d: f"{d['residual'][d['mask'] == 1].max():.4f}"),
]:
    vals = [key_fn(datasets[did]) for did in DATASET_IDS]
    print(f"{label:<30} {vals[0]:>15} {vals[1]:>15} {vals[2]:>15} {vals[3]:>15}")

print()
for did in DATASET_IDS:
    d = datasets[did]
    gt_labels, scores = d["gt_labels"], d["scores"]
    predicted = (scores > d["best_thresh"]).astype(int)
    tp = ((predicted == 1) & (gt_labels == 1)).sum()
    fp = ((predicted == 1) & (gt_labels == 0)).sum()
    fn = ((predicted == 0) & (gt_labels == 1)).sum()
    tn = ((predicted == 0) & (gt_labels == 0)).sum()
    print(f"--- Scene {did}: Confusion Matrix @ best F1 ---")
    print(f"  TP={tp:,}  FP={fp:,}")
    print(f"  FN={fn:,}  TN={tn:,}")
    if (tp+fn) > 0: print(f"  Sensitivity: {tp/(tp+fn):.4f}")
    if (tn+fp) > 0: print(f"  Specificity: {tn/(tn+fp):.4f}")
    print()

### Top-K Precision & Recall (K=500,000)

In [ ]:
TOP_K = 500000

print(f"{'':=<90}")
print(f"{'TOP-K DETECTION (K = ' + f'{TOP_K:,}' + ')':^90}")
print(f"{'':=<90}")
print()
print(f"{'Metric':<30} {'Scene 1':>12} {'Scene 2':>12} {'Scene 3':>12} {'Scene 4':>12}")
print(f"{'-'*78}")

for did in DATASET_IDS:
    d = datasets[did]
    scores, gt_labels = d["scores"], d["gt_labels"]
    k = min(TOP_K, len(scores))
    top_k_indices = np.argpartition(scores, -k)[-k:]
    predicted = np.zeros_like(gt_labels)
    predicted[top_k_indices] = 1
    tp = ((predicted == 1) & (gt_labels == 1)).sum()
    fp = ((predicted == 1) & (gt_labels == 0)).sum()
    fn = ((predicted == 0) & (gt_labels == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
    d["topk_precision"] = precision; d["topk_recall"] = recall; d["topk_f1"] = f1
    d["topk_tp"] = tp; d["topk_fp"] = fp; d["topk_fn"] = fn

for label, key in [("Precision", "topk_precision"), ("Recall", "topk_recall"),
                    ("F1", "topk_f1"), ("TP", "topk_tp"), ("FP", "topk_fp"),
                    ("FN", "topk_fn"), ("GT Anomaly Pixels", None)]:
    vals = []
    for did in DATASET_IDS:
        d = datasets[did]
        if key is None: vals.append(f"{d['gt'].sum():,}")
        elif isinstance(d[key], float): vals.append(f"{d[key]:.4f}")
        else: vals.append(f"{d[key]:,}")
    print(f"{label:<30} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12} {vals[3]:>12}")